# Trying out tests for my decision tree

I wrote these as simple checks with `assert`. Run all the cells. If something breaks, Python will tell us where.

In [1]:
import numpy as np
import pandas as pd
from model import *

## Gini, entropy, and gain

In [ ]:
assert np.isclose(gini(np.array([0, 0, 0, 0])), 0)
assert np.isclose(gini(np.array([0, 0, 1, 1])), .5)
assert np.isclose(entropy(np.array([0, 0, 1, 1])), 1)
assert np.isclose(entropy(np.array([1, 1, 1])), 0)
assert entropy(np.array([])) == 0
assert gain(np.array([]), np.array([])) == 0
assert np.isclose(gain(np.array([0, 0, 1, 1]), np.array([0, 0]), np.array([1, 1])), .5)
print("Impurity tests passed!")

Impurity tests passed!


## Splits

In [ ]:
x = np.array([1, 2, 8, 9])
y = np.array([0, 0, 1, 1])
t, score = best_split_numeric(x, y)
assert t == 5
assert np.isclose(score, .5)
cat, cat_score = best_split_categorical(np.array(["a", "a", "b", "b"]), y)
assert cat in ["a", "b"]
assert np.isclose(cat_score, .5)
df = pd.DataFrame({"number":x, "same":["a"]*4})
col, value, score, is_cat = best_split_multiple(df, y)
assert col == "number" and not is_cat
assert best_split_numeric(np.array([3, 3, 3]), np.array([0, 1, 1]))[0] is None
print("Split tests passed!")

Split tests passed!


## Nodes and helpers

In [ ]:
leaf1, leaf2 = Node(prediction=0), Node(prediction=1)
assert leaf1.is_leaf and leaf2.is_leaf
branch = Node(feature="age", value=20, is_categorical=False, left=leaf1, right=leaf2)
assert not branch.is_leaf
assert majority_helper(np.array([1, 1, 0])) == 1
assert np.array_equal(mask_helper(np.array([1, 2, 3]), 2, False), [False, True, True])
assert np.array_equal(mask_helper(np.array(["x", "y"]), "x", True), [True, False])
try:
    Node()
    assert False, "This should raise an error"
except ValueError:
    pass
print("Node tests passed!")

Node tests passed!


## Building and predicting

In [ ]:
df = pd.DataFrame({"number":[1, 2, 8, 9]})
y = pd.Series([0, 0, 1, 1])
root = build(df, y.to_numpy(), current_depth=0)
assert np.array_equal(predict_multiple(root, df), y.to_numpy())
assert predict_single(root, df.iloc[0]) == 0
assert build(df, y.to_numpy(), 0, max_depth=0).is_leaf
assert build(df, y.to_numpy(), 0, min_samples_split=10).is_leaf
assert build(pd.DataFrame({"x":[5,5,5]}), np.array([0,1,1]), 0).is_leaf
assert len(predict_multiple(root, df.iloc[:0])) == 0
print("Tree building tests passed!")

Tree building tests passed!


## Full model

In [ ]:
model = decisiontree(max_depth=3)
try:
    model.predict(df)
    assert False, "Predict should fail before fit"
except ValueError:
    pass
model.fit(df, y)
assert np.array_equal(model.predict(df), y.to_numpy())
model_entropy = decisiontree(max_depth=3, method=entropy).fit(df, y)
assert np.array_equal(model_entropy.predict(df), y.to_numpy())
try:
    decisiontree().fit(df.iloc[:0], y.iloc[:0])
    assert False, "Empty data should fail"
except ValueError:
    pass
print("Model tests passed!")

Model tests passed!


## Categorical data and multiclass labels

In [ ]:
cats = pd.DataFrame({"color":["red", "red", "blue", "blue", "green", "green"]})
labels = pd.Series(["A", "A", "B", "B", "C", "C"])
model = decisiontree().fit(cats, labels)
assert np.array_equal(model.predict(cats), labels.to_numpy())
print("Categorical and multiclass tests passed!")

Categorical and multiclass tests passed!


## Checking it on Iris

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
iris = load_iris(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=.25, random_state=42, stratify=iris.target)
model = decisiontree(max_depth=4).fit(X_train, y_train)
accuracy = accuracy_score(y_test, model.predict(X_test))
print("Iris test accuracy:", round(accuracy, 3))
assert accuracy > .7
print("All checks passed!")

Iris test accuracy: 0.921
All checks passed!
